# Tách vocal và instrumental bằng UVR5

Notebook này chuyên dùng để tách một file bài hát local trên máy thành:

- `vocal`: giọng hát / lời hát dạng audio
- `instrumental`: nhạc nền / beat

Notebook gọi trực tiếp logic UVR5 gốc của RVC:

```python
uvr(model_name, inp_root, save_root_vocal, paths, save_root_ins, agg, format0)
```

Vì vậy nếu dùng cùng model, cùng `agg`, cùng định dạng output và cùng weight thì kết quả xử lý lõi tương đương repo gốc.

## 1. Ý nghĩa các tham số UVR5

| Tham số | Ý nghĩa |
|---|---|
| `model_name` | Tên model UVR5 trong `assets/uvr5_weights`, không kèm `.pth`. Ví dụ `HP2_all_vocals`. |
| `inp_root` | Thư mục input. Repo gốc sẽ tự duyệt tất cả file trong thư mục này. Trong notebook này sẽ tạo một thư mục tạm chứa đúng một file bạn truyền vào. |
| `save_root_vocal` | Thư mục lưu vocal output. |
| `paths` | Danh sách file upload khi không dùng `inp_root`. Notebook này dùng `inp_root`, nên truyền `[]`. |
| `save_root_ins` | Thư mục lưu instrumental output. |
| `agg` | Aggressiveness của mask, thường 0-20. Cao hơn tách mạnh hơn nhưng dễ artifact hơn. |
| `format0` | Định dạng output: `wav`, `flac`, `mp3`, `m4a`, ... tùy FFmpeg hỗ trợ. |

Logic gốc sẽ kiểm tra input. Nếu audio không phải stereo `44100 Hz`, nó sẽ dùng FFmpeg để reformat về `pcm_s16le`, 2 channels, `44100 Hz` trước khi tách.

## 2. Cấu hình file input và tham số xử lý

Sửa `input_audio_path` thành đường dẫn bài hát trên máy của bạn.

In [ ]:
from pathlib import Path

# TODO: sửa đường dẫn này thành file bài hát của bạn.
input_audio_path = Path(r"D:\path\to\song.wav")

# Model mặc định nên dùng để tách toàn bộ vocal khỏi instrumental.
model_name = "HP2_all_vocals"

# Aggressiveness 0-20. 10 là mức cân bằng thường dùng.
agg = 10

# Output format. WAV dễ kiểm tra nhất, FLAC nhỏ hơn mà vẫn lossless.
output_format = "wav"

# Thư mục output nằm trong rvc_standalone/uvr_outputs nếu dùng notebook từ rvc_standalone.
output_root = Path("uvr_outputs")

# Nếu True, giữ lại thư mục input tạm để debug. Bình thường để False.
keep_temp_input = False


## 3. Bootstrap môi trường RVC/UVR5

Cell này tự tìm thư mục `rvc_standalone`, đặt `cwd`, `sys.path`, `TEMP` và `weight_uvr5_root` giống cách repo gốc cần.

In [ ]:
import os
import shutil
import sys
import uuid
from IPython.display import Audio, display


def find_rvc_standalone_root() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / "rvc_standalone",
        cwd.parent / "rvc_standalone",
    ]
    for candidate in candidates:
        if (candidate / "infer" / "modules" / "uvr5" / "modules.py").is_file():
            return candidate.resolve()
    raise RuntimeError("Không tìm thấy thư mục rvc_standalone chứa infer/modules/uvr5/modules.py")


project_root = find_rvc_standalone_root()
os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

assets_root = project_root / "assets"
uvr_weight_root = assets_root / "uvr5_weights"
temp_root = project_root / "TEMP" / "uvr_notebook"
temp_root.mkdir(parents=True, exist_ok=True)

os.environ["weight_uvr5_root"] = str(uvr_weight_root)
os.environ["TEMP"] = str(temp_root)

# Config của RVC dùng argparse.parse_args(). Trong notebook, sys.argv thường có
# tham số riêng của Jupyter nên cần sanitize trước khi import uvr.
sys.argv = ["rvc_uvr_notebook"]

print("Project root:", project_root)
print("UVR5 weights:", uvr_weight_root)
print("TEMP:", temp_root)


## 4. Kiểm tra model UVR5 đã có chưa

Nếu danh sách rỗng hoặc thiếu model bạn chọn, cần tải UVR5 weights trước. Trong repo gốc có thể dùng `tools/download_models.py` hoặc script tải asset tương ứng.

In [ ]:
available_models = sorted(path.stem for path in uvr_weight_root.glob("*.pth"))
print("Các model .pth đang có:")
for name in available_models:
    print("-", name)

model_path = uvr_weight_root / f"{model_name}.pth"
if not model_path.is_file():
    raise FileNotFoundError(
        f"Không tìm thấy model {model_name}.pth tại {model_path}. "
        "Hãy tải UVR5 weights trước hoặc đổi model_name sang model đang có."
    )

if not input_audio_path.is_file():
    raise FileNotFoundError(f"Không tìm thấy input audio: {input_audio_path}")

print("\nInput:", input_audio_path.resolve())
print("Model:", model_path.resolve())


## 5. Xem thông tin input audio

Cell này dùng `ffprobe` để xem duration, sample rate, channel trước khi tách.

In [ ]:
import json
import subprocess


def ffprobe_audio(path: Path) -> dict:
    cmd = [
        "ffprobe",
        "-v", "error",
        "-print_format", "json",
        "-show_streams",
        "-show_format",
        str(path),
    ]
    proc = subprocess.run(cmd, text=True, capture_output=True)
    if proc.returncode != 0:
        raise RuntimeError(proc.stderr.strip() or proc.stdout.strip())
    data = json.loads(proc.stdout)
    stream = next(s for s in data["streams"] if s.get("codec_type") == "audio")
    fmt = data.get("format", {})
    return {
        "duration_sec": float(fmt.get("duration", 0.0)),
        "sample_rate": int(stream.get("sample_rate", 0)),
        "channels": int(stream.get("channels", 0)),
        "channel_layout": stream.get("channel_layout", ""),
        "codec": stream.get("codec_name", ""),
        "format": fmt.get("format_name", ""),
        "size_mb": round(path.stat().st_size / (1024 * 1024), 3),
    }


ffprobe_audio(input_audio_path)


## 6. Chạy tách vocal/instrumental

Để dùng đúng nhánh `inp_root` của repo gốc, notebook sẽ tạo một thư mục input tạm chứa đúng một bản copy của file bạn truyền vào. Sau đó gọi:

```python
uvr(model_name, inp_root, save_root_vocal, [], save_root_ins, agg, output_format)
```

In [ ]:
from infer.modules.uvr5.modules import uvr


def safe_file_name(path: Path) -> str:
    keep = []
    for ch in path.name:
        keep.append(ch if ch.isascii() and (ch.isalnum() or ch in "._-") else "_")
    return "".join(keep).strip("._-") or "input.wav"


job_id = uuid.uuid4().hex[:12]
job_root = output_root / f"uvr_{input_audio_path.stem}_{job_id}"
inp_root = job_root / "input"
save_root_vocal = job_root / "vocal"
save_root_ins = job_root / "instrumental"

for directory in (inp_root, save_root_vocal, save_root_ins):
    directory.mkdir(parents=True, exist_ok=True)

local_input = inp_root / safe_file_name(input_audio_path)
shutil.copy2(input_audio_path, local_input)

print("inp_root:", inp_root.resolve())
print("save_root_vocal:", save_root_vocal.resolve())
print("save_root_ins:", save_root_ins.resolve())

logs = []
for info in uvr(
    model_name,
    str(inp_root),
    str(save_root_vocal),
    [],
    str(save_root_ins),
    agg,
    output_format,
):
    logs.append(info)
    print(info)

if not keep_temp_input:
    shutil.rmtree(inp_root, ignore_errors=True)

print("\nHoàn tất. Output root:", job_root.resolve())


## 7. Lấy đường dẫn output và nghe thử

Tên file output do UVR5 gốc tạo, thường có dạng:

- `vocal_<tên file>_<agg>.<format>`
- `instrument_<tên file>_<agg>.<format>`

Một số model đặc biệt như `HP3` có thể đảo prefix theo logic gốc.

In [ ]:
vocal_outputs = sorted(path for path in save_root_vocal.iterdir() if path.is_file())
instrumental_outputs = sorted(path for path in save_root_ins.iterdir() if path.is_file())

if not vocal_outputs:
    raise RuntimeError(f"Không có file vocal output trong {save_root_vocal}")
if not instrumental_outputs:
    raise RuntimeError(f"Không có file instrumental output trong {save_root_ins}")

vocal_output = vocal_outputs[0]
instrumental_output = instrumental_outputs[0]

print("Vocal output:", vocal_output.resolve())
print("Instrumental output:", instrumental_output.resolve())

print("\nThông tin vocal:")
print(ffprobe_audio(vocal_output))

print("\nThông tin instrumental:")
print(ffprobe_audio(instrumental_output))


In [ ]:
print("Nghe vocal:")
display(Audio(str(vocal_output)))

print("Nghe instrumental:")
display(Audio(str(instrumental_output)))


## 8. Gợi ý chọn model

- `HP2_all_vocals`: tách toàn bộ vocal khỏi instrumental, lựa chọn mặc định tốt cho bài hát.
- `HP5_only_main_vocal`: ưu tiên giọng chính, có thể để backing vocal hoặc một số âm thanh khác ở instrumental.
- `VR-DeEchoNormal`, `VR-DeEchoAggressive`, `VR-DeEchoDeReverb`: nhóm model xử lý echo/reverb, không phải lựa chọn mặc định để tách vocal/instrumental thông thường.
- `HP3_all_vocals`: trong logic gốc có xử lý prefix output đặc biệt qua `is_hp3`.

Nếu vocal còn dính nhạc nền, thử tăng `agg` lên 12-15. Nếu vocal bị méo hoặc mất chi tiết, giảm `agg` xuống 5-8.